# Stage 1: Dataset Inventory & Pre-flight Inspection
**Project:** Machine Learning-Based Network Intrusion Detection System (ML-NIDS) with Random Forest and SHAP Explainable AI  
**Objective:** Verify archive presence, inspect binary file structures, validate schemas, count records, measure ground-truth class distributions, and generate the dataset manifest.

In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd

# Set project root
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

from src.data.inventory import generate_manifest
from src.data.schema import UNIFLOW_SCHEMA, BIFLOW_SCHEMA, PACKET_SCHEMA

## 1. Execute Dataset Inventory Pipeline
This runs the streaming inspection script directly across all four ZIP archives without expanding gigabytes to disk.

In [ ]:
manifest = generate_manifest(project_root)

## 2. Inspect Flow Class Distributions & Proportions

In [ ]:
rows = []
for arch_key in ["uniflow_features", "biflow_features"]:
    files = manifest["archives"][arch_key]["files"]
    for fname, finfo in files.items():
        cd = finfo["class_distribution"]
        rows.append({
            "Type": "Uniflow" if "uniflow" in arch_key else "Biflow",
            "Scenario File": fname,
            "Total Records": finfo["total_records"],
            "Normal (0)": cd.get("normal_0", 0),
            "Attack (1)": cd.get("attack_1", 0),
            "Attack %": cd.get("attack_pct", 0.0)
        })

df_summary = pd.DataFrame(rows)
display(df_summary)

## 3. Key Observations & Anomaly Resolution
- Notice that `uniflow_scan_A.csv`, `uniflow_scan_sU.csv`, `uniflow_mqtt_bruteforce.csv`, and `uniflow_sparta.csv` contain mixed traffic (both normal background traffic and attack traffic).
- Files are **not** naively labeled by filename. They contain genuine per-flow ground-truth labels.